In [1]:
import pandas as pd
amazon=pd.read_parquet("parquet/amazon_new.parquet")
goodreads=pd.read_parquet("parquet/goodreads_new.parquet")
# We are dropping columns that are not needed for data fusion. These two columns only exist in goodreads dataset
metabooks=pd.read_parquet("parquet/metabooks_new.parquet")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)



In [2]:
display(amazon.head(1).style.set_caption("Amazon"))
display(goodreads.head(1).style.set_caption("Goodreads"))
display(metabooks.head(1).style.set_caption("Metabooks"))


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,amazon_1,classical mythology,mark p o morford,None,None,None,None,None,None,None,Oxford University Press,2002,None,0195153448


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,goodreads_1,the hunger games,suzanne collins,4.330000,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",Hardcover,First Edition,374,Scholastic Press,2008,5.090000,0439023483


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,metabooks_1,alvis the story of the red triangle,kenneth day,4.100000,3,English,"['Engineering', 'Transportation', 'Automotive']",None,None,400,Haynes Pubns,,112.300003,1844255247


In [3]:
common_isbns = list(
    set(amazon['isbn_clean'])
    & set(goodreads['isbn_clean'])
    & set(metabooks['isbn_clean'])
)

In [4]:
goodreads.columns

Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

In [5]:
amazon.columns

Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

In [6]:
import pandas as pd, numpy as np, re, ast


# Union schema we care about
BASE_COLS = [
    "id","title","author","publisher","publish_year","language",
    "numratings","genres","page_count","price","isbn_clean"
]

def _add_missing_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def build_longform(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Normalize inputs to the same schema, add source label, and stack them."""
    amz = amazon.copy()
    gr  = goodreads.copy()
    mb  = metabooks.copy()

    amz["source"] = "amazon"
    gr["source"]  = "goodreads"
    mb["source"]  = "metabooks"

    def keep_cols(df): 
        return df[[c for c in BASE_COLS if c in df.columns] + ["source"]]
    
    long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)

    # normalize empties to NaN
    for c in ["title","author","publisher","language","genres","isbn_clean","id"]:
        if c in long.columns: 
            long[c] = long[c].replace({"": np.nan})
    return long


def fuse_title(g: pd.DataFrame):
    """
    Choose the LONGEST non-null title as-is (no cleaning).
    Tie-breakers: more tokens, then alphabetical.
    """
    vals = g["title"].dropna().astype(str)
    if vals.empty:
        return None
    vals = vals.map(lambda s: s.strip())
    return max(vals, key=lambda s: (len(s), len(s.split()), s))


def fuse_author(g: pd.DataFrame):
    """
    Prefer Amazon + Metabooks authors; pick the LONGEST string as-is (no cleaning).
    If none available, fallback to Goodreads. Tie-breakers: more tokens, then alphabetical.
    """
    # Prefer amazon + metabooks
    sub = g[g["source"].isin(["amazon", "metabooks"])]["author"].dropna().astype(str).map(str.strip)
    if len(sub):
        return max(sub, key=lambda s: (len(s), len(s.split()), s))

    # Fallback: goodreads
    gr = g[g["source"].eq("goodreads")]["author"].dropna().astype(str).map(str.strip)
    if len(gr):
        return max(gr, key=lambda s: (len(s), len(s.split()), s))

    return None


# ---------- publisher (prefer Metabooks > Goodreads > Amazon) ----------
def fuse_publisher(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        v = g.loc[g["source"].eq(src), "publisher"].dropna()
        if len(v):
            return str(v.iloc[0]).strip()
    return None


# ---------- publish_year (prefer Metabooks > Goodreads > Amazon; guard range) ----------
def fuse_publish_year(g: pd.DataFrame, year_min=1400, year_max=2100):
    for src in ["metabooks","goodreads","amazon"]:
        v = pd.to_numeric(g.loc[g["source"].eq(src), "publish_year"], errors="coerce").dropna()
        if len(v):
            y = int(v.iloc[0])
            if year_min <= y <= year_max:
                return y
    return None


# ---------- rating (prefer Metabooks > Goodreads > Amazon) ----------
def fuse_rating(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        vv = pd.to_numeric(g.loc[g["source"].eq(src), "rating"], errors="coerce").dropna()
        if len(vv):
            return float(vv.iloc[0])
    return None


# ---------- numratings (keep it the same -> align with rating source; MB > GR > AZ) ----------
def fuse_numratings(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        vv = pd.to_numeric(g.loc[g["source"].eq(src), "numratings"], errors="coerce").dropna()
        if len(vv):
            return int(vv.iloc[0])
    return None


# ---------- genres (union of unique genres from GR + MB) ----------
def _parse_genres(s):
    if pd.isna(s): return []
    s = str(s).strip()
    try:
        lst = ast.literal_eval(s)
        if isinstance(lst, list): 
            return [str(x) for x in lst]
    except Exception:
        pass
    return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]

def fuse_genres(g: pd.DataFrame, as_list=False):
    sub = g[g["source"].isin(["goodreads","metabooks"])]
    all_genres = []
    for s in sub["genres"].dropna():
        all_genres.extend(_parse_genres(s))
    uniq = sorted({x.strip().title() for x in all_genres if x})
    return uniq if as_list else (", ".join(uniq) if uniq else None)


# ---------- page_count (take Metabooks as the SOLE source) ----------
def fuse_page_count(g: pd.DataFrame, min_pages=20):
    """
    Prefer Metabooks page_count; if missing, use Goodreads.
    Do NOT use Amazon (it doesn't provide page_count).
    """
    # Metabooks
    mb = pd.to_numeric(g.loc[g["source"].eq("metabooks"), "page_count"], errors="coerce").dropna()
    mb = mb[mb >= min_pages]
    if len(mb):
        return int(round(float(mb.iloc[0])))

    # Goodreads (fallback)
    gr = pd.to_numeric(g.loc[g["source"].eq("goodreads"), "page_count"], errors="coerce").dropna()
    gr = gr[gr >= min_pages]
    if len(gr):
        return int(round(float(gr.iloc[0])))

    return None


# ---------- price (prefer Metabooks > Goodreads > Amazon) ----------
def fuse_price(g: pd.DataFrame):
    """
    Prefer Metabooks price; if missing/NaN or non-positive, use Goodreads.
    Ignore Amazon entirely (column is artificial and NaN-filled).
    """
    sub = g[g["source"].isin(["metabooks", "goodreads"])][["source", "price"]].copy()
    sub["price"] = pd.to_numeric(sub["price"], errors="coerce")

    # Metabooks first
    mb = sub.loc[sub["source"].eq("metabooks"), "price"].dropna()
    mb = mb[mb > 0]
    if len(mb):
        return float(round(mb.iloc[0], 2))

    # Goodreads fallback
    gr = sub.loc[sub["source"].eq("goodreads"), "price"].dropna()
    gr = gr[gr > 0]
    if len(gr):
        return float(round(gr.iloc[0], 2))

    return None

def fuse_language(g: pd.DataFrame):
    """
    Prefer Metabooks language; if missing/NaN/empty, use Goodreads.
    Ignore Amazon entirely. Safe if 'language' column is absent.
    """
    if "language" not in g.columns:
        return None

    sub = g[g["source"].isin(["metabooks", "goodreads"])][["source", "language"]].copy()
    sub["language"] = sub["language"].astype(str).str.strip()
    sub.loc[sub["language"].str.lower().isin(["", "nan", "none"]), "language"] = np.nan

    mb = sub.loc[sub["source"].eq("metabooks"), "language"].dropna()
    if len(mb):
        return mb.iloc[0]
    gr = sub.loc[sub["source"].eq("goodreads"), "language"].dropna()
    if len(gr):
        return gr.iloc[0]
    return None


# ---------- Main fusion pipeline ----------
def fuse_all(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Fuse datasets, keeping only ISBNs present in at least two sources."""
    long = build_longform(amazon, goodreads, metabooks)

    # ✅ Keep only ISBNs that appear in at least two different sources
    long = long[long.groupby("isbn_clean")["source"].transform("nunique") >= 2]

    fused = (long.groupby("isbn_clean", as_index=False)
                  .apply(lambda g: pd.Series({
                      "title":         fuse_title(g),
                      "author":        fuse_author(g),
                      "publisher":     fuse_publisher(g),
                      "publish_year":  fuse_publish_year(g),
                      "language":      fuse_language(g),
                      "numratings":    fuse_numratings(g),
                      "genres":        fuse_genres(g, as_list=False),
                      "page_count":    fuse_page_count(g),
                      "price":         fuse_price(g),
                  }))
                  .reset_index(drop=True))

    # Dtype tidy-up (cast only when safe)
    for col, dtype in [("publish_year","Int16"), ("page_count","Int32"), ("numratings","Int64")]:
        fused[col] = pd.to_numeric(fused[col], errors="ignore").astype(dtype, errors="ignore")

    # Column order
    fused = fused[["isbn_clean","title","author","publisher","publish_year","language",
                   "numratings","genres","page_count","price"]]
    return fused

In [7]:
fused_df = fuse_all(amazon, goodreads, metabooks)

/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_99116/63143198.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)
/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_99116/63143198.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)
/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_99116/63143198.py:2

In [8]:
fused_df.shape

(38828, 10)

In [9]:
fused_df.isnull().sum()

isbn_clean         0
title              0
author             0
publisher          0
publish_year     120
language         133
numratings         0
genres          1380
page_count      3417
price           2388
dtype: int64

In [10]:
fused_df.to_parquet("ml-datasets/golden_fused_books.parquet", index=False)

In [11]:
fused_df.head()

,isbn_clean,title,author,publisher,publish_year,language,numratings,genres,page_count,price
0,0000913154,the way things work an illustrated encyclopedia of technology,c van amerongen translator,Simon & Schuster,1967,English,44,"Engineering, Transportation",592,26.00
1,000100039X,the prophet,kahlil gibran,Rupa (Educa Books),2003,English,12423,"Classics, Fiction, Inspirational, Literature, Novels, Philosophy, Poetry, Religion, Self Help, Spirituality",144,6.00
2,0001048082,made in america,bill bryson,HarperCollins Audio Books,1995,English,13137,"American, American History, Audiobook, History, Humor, Language, Linguistics, Nonfiction, The United States Of America, Travel",<NA>,3.81
3,0001837397,autumn story introduce children to the seasons in the gorgeously illustrated classics of brambly hedge,jill barklem,HarperCollinsChildren’sBooks,<NA>,English,200,"Children'S Books, Early Learning",32,9.99
4,0002113570,in the shadow of man,jane goodall,Collins,1971,English,764,"Biological Sciences, Math, Science",256,6.45
